In [1]:
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".08"
import numpy as np
import tifffile
from skophys.preprocessing import Vectorizer, UnVectorizer
from skophys.sm import NNCCA, NNSM
from skophys.datasets import ARProcessMovie
from sklearn.metrics.pairwise import cosine_similarity
from skimage.transform import downscale_local_mean
from scipy.optimize import linear_sum_assignment
from ipywidgets import HBox, VBox
import fastplotlib as fpl
from matplotlib import pyplot as plt
from tqdm import tqdm
from pathlib import Path
import seaborn as sns
from itertools import product

fastplotlib version from git (0.5.0) and __version__ (0.5.1) don't match.
rendercanvas version from git (2.0.0) and __version__ (2.0.3) don't match.
Unable to find extension: VK_EXT_swapchain_colorspace


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),AMD RADV POLARIS10 (ACO),DiscreteGPU,Vulkan,Mesa 20.3.5 (ACO)
❗ limited,"llvmpipe (LLVM 11.0.1, 256 bits)",CPU,Vulkan,Mesa 20.3.5 (LLVM 11.0.1)
✅,NVIDIA GeForce RTX 3080,DiscreteGPU,Vulkan,530.30.02
❌,"Radeon RX 570 Series (POLARIS10, DRM 3.40.0, 5.10.0-21-amd64, LLVM 11.0.1)",Unknown,OpenGL,4.6 (Core Profile) Mesa 20.3.5


pygfx version from git (0.9.0) and __version__ (0.10.0) don't match.


In [2]:
%matplotlib inline

In [3]:
import pandas as pd

In [4]:
datasets = ["simple", "complex"]
noise_options = [0.1, 0.5, 0.8, 1, 1.5]
methods = {"nnSM": NNSM, "nnCCA": NNCCA}
lags = [1, 2]

df = pd.DataFrame(columns=["dataset", "noise", "method", "lag", "cs_trace"])

for dataset in datasets:
    for noise_sigma in noise_options:
        for method in methods.keys():
            for lag in lags:
                if dataset == "simple":
                    locs = np.array([
                        [0, 0],
                        [2, 0]
                    ])
                    
                    ar_model = ARProcessMovie(
                        n_timepoints=5000,
                        n_components=2,
                        ar_noise_sigma=0.1,
                        obs_noise_sigma=noise_sigma,
                        movie_dims=(10, 15),
                        component_locs=locs,
                        firing_probability=0.02
                    )
                    
                    movie = ar_model.movie
                elif dataset == "complex":
                    locs = np.array([
                        [0, 0],
                        [2, 0],
                        [4, 0],
                        [0, 2],
                        [5, 5],
                        [3, 3],
                        [0, 5],
                    ])
                    
                    ar_model = ARProcessMovie(
                        n_timepoints=5000,
                        n_components=locs.shape[0],
                        random_seed=0,
                        ar_noise_sigma=0.1,
                        obs_noise_sigma=noise_sigma,
                        movie_dims=(15, 15),
                        component_locs=locs,
                        firing_probability=0.01
                    )
                    
                    movie = ar_model.movie

                model = methods[method](lag=lag, lag_step=1)

                n_init = 1_000

                model.pre_initialize(movie[:n_init], max_k=ar_model.traces.shape[0], k=ar_model.traces.shape[0])
                
                err_log, Ys, Ws = model.initialize(max_iter=5_000, keep_iteration_results=True, eta=0.1, inertia=1e-3, error_threshold=0.01)
                
                permute_ixs, cos_sim = model.permute_indices(ar_model.traces[:, : model.init_arrays.Y.shape[1]])
                
                if model.lag > 1:
                    permute_ixs, cos_sim = model.permute_indices(ar_model.traces[:, 1:n_init - model.lag])
                
                tr = np.trace(cos_sim)

                df.loc[df.index.size] = pd.Series(
                    {
                        "dataset": dataset,
                        "noise": noise_sigma,
                        "method": method,
                        "lag": lag,
                        "cs_trace": tr / ar_model.traces.shape[0]
                    }
                )

creating H
computing covariance
truncated whitening
performing nnSVD


  2%|█▏                                                                     | 84/5000 [00:01<01:17, 63.17it/s]


creating H
computing covariance
truncated whitening
performing nnSVD


  1%|▉                                                                      | 67/5000 [00:00<00:53, 91.76it/s]


creating H
Whitening H
computing covariance
performing nnSVD


  1%|▊                                                                    | 62/5000 [00:00<00:04, 1078.11it/s]


... (73 outputs left out)

creating H
Whitening H
computing covariance
performing nnSVD


100%|████████████████████████████████████████████████████████████████████| 5000/5000 [00:05<00:00, 896.03it/s]


creating H
Whitening H
computing covariance
performing nnSVD


100%|████████████████████████████████████████████████████████████████████| 5000/5000 [00:05<00:00, 873.15it/s]


In [5]:
df

,dataset,noise,method,lag,cs_trace
0,simple,0.1,nnSM,1,0.950599
1,simple,0.1,nnSM,2,0.908833
2,simple,0.1,nnCCA,1,0.906862
3,simple,0.1,nnCCA,2,0.860224
4,simple,0.5,nnSM,1,0.948874
5,simple,0.5,nnSM,2,0.908905
6,simple,0.5,nnCCA,1,0.904961
7,simple,0.5,nnCCA,2,0.534998
8,simple,0.8,nnSM,1,0.943038
9,simple,0.8,nnSM,2,0.908304
